# [LAB 05] XGBoost Channel 분류 및 TabPy 배포
## 구매 패턴 기반 고객 군집 노트북과 동일 전처리 → Channel(1/2) 예측

- **전처리**: LAB 05와 동일 (np.log1p → StandardScaler, 6개 로그 스케일 컬럼 사용)
- **타겟**: Channel (1 또는 2) → 라벨 0/1로 변환하여 이진 분류
- **모델**: XGBoost 분류기
- **저장**: scaler, XGB 모델 각각 pkl 저장 → TabPy/Tableau 시뮬레이터 연동

## 1. 라이브러리 및 데이터 로드

In [ ]:
from hossam import load_data
import pandas as pd
import numpy as np
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
import xgboost as xgb
import pickle
from pathlib import Path

my_dpi = 200
RAW_COLS = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
LAB05_DIR = Path('.').resolve()

In [ ]:
origin = load_data('wholesale_customers')
df = origin[RAW_COLS + ['Channel']].copy()
df.head(10)

## 2. 전처리 (LAB 05와 동일: log1p → StandardScaler)

1. 원본 6컬럼에 **np.log1p** 적용하여 log_ 컬럼 추가  
2. **StandardScaler**로 12컬럼(원본+log) 표준화  
3. 학습에는 **스케일된 log 6컬럼**만 사용 (기존 PCA 입력과 동일)

In [ ]:
df_log = df[RAW_COLS].copy()
for c in RAW_COLS:
    df_log[f'log_{c}'] = np.log1p(df_log[c])

scaler = StandardScaler()
sdf = DataFrame(scaler.fit_transform(df_log), columns=df_log.columns)
log_cols = [f'log_{c}' for c in RAW_COLS]
X = sdf[log_cols].values
y = (df['Channel'] - 1).values  # 1,2 → 0,1

print('X shape:', X.shape)
print('y unique:', np.unique(y))
X[:3]

## 3. train/test split 및 XGBoost 학습

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=52, stratify=y
)

clf = xgb.XGBClassifier(random_state=52, use_label_encoder=False, eval_metric='logloss')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)
y_pred_proba_1 = y_pred_proba[:, 1]  # 양성(Channel=2) 확률

## 4. 성능 지표: Accuracy, Confusion Matrix, ROC-AUC

In [ ]:
acc = accuracy_score(y_test, y_pred)
print('Accuracy:', acc)
print('Precision:', precision_score(y_test, y_pred, zero_division=0))
print('Recall:', recall_score(y_test, y_pred, zero_division=0))
print('F1:', f1_score(y_test, y_pred, zero_division=0))
print('ROC-AUC:', roc_auc_score(y_test, y_pred_proba_1))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cmdf = DataFrame(
    cm,
    index=['Actual 0 (Channel 1)', 'Actual 1 (Channel 2)'],
    columns=['Pred 0', 'Pred 1']
)
fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=my_dpi)
sb.heatmap(cmdf, annot=True, fmt='d', linewidths=0.5, cmap='PuOr')
ax.set_title('Confusion Matrix (Channel 예측)')
plt.tight_layout()
plt.show()
plt.close()
cmdf

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_1)
auc = roc_auc_score(y_test, y_pred_proba_1)
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=my_dpi)
ax.plot(fpr, tpr)
ax.plot([0, 1], [0, 1], color='red', linestyle=':', alpha=0.5)
ax.fill_between(fpr, tpr, alpha=0.1)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title(f'ROC Curve (AUC={auc:.4f})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

## 5. Feature Importance

In [ ]:
imp = DataFrame({
    'feature': log_cols,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)
imp

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=my_dpi)
sb.barplot(data=imp, x='importance', y='feature', palette='viridis')
ax.set_title('XGBoost Feature Importance (Channel 예측)')
plt.tight_layout()
plt.show()
plt.close()

## 6. Scaler / XGB 모델 저장 (TabPy·Tableau 시뮬레이터용)

In [ ]:
scaler_path = LAB05_DIR / 'wholesale_scaler.pkl'
model_path = LAB05_DIR / 'wholesale_xgb.pkl'

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
with open(model_path, 'wb') as f:
    pickle.dump(clf, f)

print('저장 완료:', scaler_path)
print('저장 완료:', model_path)

## 7. TabPy 배포용 함수: 원본 수치 → 예측 클래스 & 확률

Tableau 시뮬레이터에서 **원본 구매액(Raw)** 6개를 넘기면,  
1. np.log1p 적용  
2. 저장된 StandardScaler로 스케일  
3. XGBoost predict_proba  
4. **예측 클래스(0/1)** 와 **양성(Channel=2) 확률** 을 리스트로 반환

In [ ]:
def predict_customer_channel(Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen):
    """
    TabPy/Tableau에서 호출할 함수.
    인자: 원본 구매액 6개 (각각 리스트 또는 단일 값)
    반환: (예측 클래스 리스트, 양성 확률 리스트)
    """
    def to_arr(x):
        if hasattr(x, '__len__') and not isinstance(x, (str, bytes)):
            return np.asarray(x, dtype=float)
        return np.asarray([float(x)], dtype=float)

    fresh = to_arr(Fresh)
    milk = to_arr(Milk)
    grocery = to_arr(Grocery)
    frozen = to_arr(Frozen)
    det = to_arr(Detergents_Paper)
    deli = to_arr(Delicassen)

    n = max(len(fresh), len(milk), len(grocery), len(frozen), len(det), len(deli))
    if n == 0:
        return [], []
    if len(fresh) == 1 and n > 1: fresh = np.full(n, float(fresh.flat[0]))
    if len(milk) == 1 and n > 1: milk = np.full(n, float(milk.flat[0]))
    if len(grocery) == 1 and n > 1: grocery = np.full(n, float(grocery.flat[0]))
    if len(frozen) == 1 and n > 1: frozen = np.full(n, float(frozen.flat[0]))
    if len(det) == 1 and n > 1: det = np.full(n, float(det.flat[0]))
    if len(deli) == 1 and n > 1: deli = np.full(n, float(deli.flat[0]))

    raw = np.column_stack([fresh, milk, grocery, frozen, det, deli])
    log = np.log1p(raw)
    full = np.hstack([raw, log])
    scaled = scaler.transform(full)
    X_in = scaled[:, 6:12]

    proba = clf.predict_proba(X_in)
    pred_class = clf.predict(X_in)
    proba_positive = proba[:, 1].tolist()

    return pred_class.tolist(), proba_positive

In [ ]:
# 검증: 테스트 데이터 몇 개로 호출
raw_test = origin[RAW_COLS].iloc[:5].values
p, prob = predict_customer_channel(
    raw_test[:, 0], raw_test[:, 1], raw_test[:, 2],
    raw_test[:, 3], raw_test[:, 4], raw_test[:, 5]
)
print('예측 클래스:', p)
print('Channel=2 확률:', prob)
print('실제 Channel:', (origin['Channel'].iloc[:5] - 1).tolist())